## Step 1: Project Initialization and Dependency Setup

In [1]:
!pip install groq chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Data Loading and Preprocessing

**Fix 1 (Label alignment):** We now parse `anomaly_types`, `latency`, and `block_id` directly
from the RAG document text — these are the fields generated by LAD4. This ensures the metadata
stored in ChromaDB comes from the actual document, not from a separate join that could mismatch.

In [3]:
import pandas as pd
import re

rag_df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/results/rag_documents.csv")
documents = rag_df["RAG_Document"].tolist()

print("Total RAG documents:", len(documents))
print("\nSample document:")
print(documents[0])

Total RAG documents: 3378

Sample document:
Block ID        : blk_8462687553742484299
Label           : Fail
Anomaly Type(s) : duplicate_pattern

--- Details ---
Event Sequence  : E5 -> E5 -> E5 -> E22 -> E11 -> E9 -> E11 -> E9 -> E11 -> E9 -> E26 -> E26 -> E26 -> E23 -> E23 -> E23 -> E21 -> E20 -> E21 -> E21
Readable Form   : [*]Receiving block[*]src:[*]dest:[*] -> [*]Receiving block[*]src:[*]dest:[*] -> [*]Receiving block[*]src:[*]dest:[*] -> [*]BLOCK* NameSystem[*]allocateBlock:[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is a

In [4]:
# ── FIX 1 & 3: Parse structured metadata from each RAG document ──────────────
# Instead of relying on a separate CSV join, we extract all metadata fields
# directly from the RAG document text. This means block_id, anomaly_types and
# latency stored in ChromaDB come from the same source as the document content.

def parse_metadata_from_doc(doc: str) -> dict:
    """
    Extract structured metadata fields from a RAG document string.
    Returns a dict with block_id, anomaly_types (primary), latency, total_events.
    """
    block_id_match  = re.search(r'Block ID\s*:\s*(\S+)', doc)
    label_match     = re.search(r'^Label\s*:\s*(.+)', doc, re.MULTILINE)
    atypes_match    = re.search(r'Anomaly Type\(s\)\s*:\s*(.+)', doc)
    latency_match   = re.search(r'Latency\s*:\s*(\d+)', doc)
    total_ev_match  = re.search(r'Total Events\s*:\s*(\d+)', doc)

    full_types = atypes_match.group(1).strip() if atypes_match else 'unknown'
    # Primary type = first type listed (used for ChromaDB metadata filter)
    primary_type = full_types.split(',')[0].strip()

    return {
        "block_id":      block_id_match.group(1).strip() if block_id_match else 'unknown',
        "label":         label_match.group(1).strip()    if label_match    else 'Fail',
        "anomaly_types": full_types,
        "primary_type":  primary_type,
        "latency":       int(latency_match.group(1))     if latency_match  else 0,
        "total_events":  int(total_ev_match.group(1))    if total_ev_match else 0,
    }

# Parse metadata for all documents
all_metadata = [parse_metadata_from_doc(doc) for doc in documents]

# Show distribution of anomaly types
import pandas as pd
meta_df = pd.DataFrame(all_metadata)
print("Anomaly type distribution:")
print(meta_df['primary_type'].value_counts().to_string())
print("\nSample parsed metadata:")
print(all_metadata[0])

Anomaly type distribution:
primary_type
duplicate_pattern    1521
missing_events       1226
repetition            587
high_latency           44

Sample parsed metadata:
{'block_id': 'blk_8462687553742484299', 'label': 'Fail', 'anomaly_types': 'duplicate_pattern', 'primary_type': 'duplicate_pattern', 'latency': 7156, 'total_events': 20}


## Step 3: Semantic Embedding Generation

In [5]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings  = embed_model.encode(documents, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

Embedding shape: (3378, 384)


## Step 4: Vector Database Indexing with ChromaDB

In [6]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.Client(
    Settings(persist_directory="./chroma_db")
)

# Delete existing collection so we can rebuild it with metadata
try:
    chroma_client.delete_collection(name="anomaly_logs")
    print("Old collection deleted — rebuilding with metadata.")
except:
    pass

collection = chroma_client.get_or_create_collection(name="anomaly_logs")

# ── FIX 3 + 4: Index documents WITH metadata ──────────────────────────────────
# ChromaDB metadata values must be str / int / float — not lists.
# We store full anomaly_types as a comma-separated string and primary_type
# as a single string for clean WHERE-clause filtering.

BATCH_SIZE = 500   # ChromaDB handles large adds better in batches

for start in range(0, len(documents), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(documents))
    collection.add(
        documents  = documents[start:end],
        embeddings = embeddings[start:end].tolist(),
        ids        = [str(i) for i in range(start, end)],
        metadatas  = all_metadata[start:end],
    )
    print(f"  Indexed {end}/{len(documents)} documents...")

print(f"\nTotal vectors in ChromaDB: {collection.count()}")

  Indexed 500/3378 documents...
  Indexed 1000/3378 documents...
  Indexed 1500/3378 documents...
  Indexed 2000/3378 documents...
  Indexed 2500/3378 documents...
  Indexed 3000/3378 documents...
  Indexed 3378/3378 documents...

Total vectors in ChromaDB: 3378


## Step 5: Retrieval Logic

In [7]:
def retrieve_similar_anomalies(
    query: str,
    k: int = 3,
    anomaly_type_filter: str = None
) -> list[dict]:
    """
    Retrieve k most semantically similar anomaly documents for a query.

    Args:
        query:               Free-text query or auto-constructed summary string.
        k:                   Number of results to retrieve.
        anomaly_type_filter: Optional primary_type to restrict search scope
                             (e.g. 'high_latency', 'repetition', 'missing_events',
                              'duplicate_pattern').

    Returns:
        List of dicts with keys: document, block_id, primary_type, distance.
    """
    query_embedding = embed_model.encode([query]).tolist()

    # ── FIX 4: Apply metadata filter when anomaly type is known ───────────────
    where_clause = None
    if anomaly_type_filter:
        where_clause = {"primary_type": {"$eq": anomaly_type_filter}}

    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = k,
        where            = where_clause,
    )

    retrieved_docs = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved_docs.append({
            "document":     doc,
            "block_id":     meta.get("block_id", "unknown"),   # FIX 3
            "primary_type": meta.get("primary_type", "unknown"),
            "latency":      meta.get("latency", 0),
            "distance":     dist,
        })

    return retrieved_docs

## Step 6: LLM Root Cause Analysis

In [ ]:
from groq import Groq

groq_client = Groq(api_key="your api key")  # <-- replace with your key

In [9]:
import json
import re
from collections import Counter

# ============================
# 🔹 EVENT DESCRIPTIONS (unchanged)
# ============================
EVENT_DESCRIPTIONS = {
    'E1':  'A DataNode received a request to store a block that already exists in its local storage — indicates duplicate write or stale block reference.',
    'E2':  'Block checksum verification passed — the data written to a DataNode is confirmed intact.',
    'E3':  'A DataNode successfully served a block read request to a client or another DataNode.',
    'E4':  'An exception occurred while a DataNode was serving a block — the read or transfer request failed.',
    'E5':  'A DataNode started receiving a new block from a source DataNode or client — normal start of a pipeline write.',
    'E6':  'A DataNode finished receiving a full block from the pipeline — confirms complete transfer.',
    'E7':  'An exception was thrown during a block write operation — the write pipeline encountered an error.',
    'E8':  'The PacketResponder thread handling block acknowledgements was interrupted — signals unexpected pipeline disruption.',
    'E9':  'A DataNode finished receiving a block of a known size — confirms successful individual replica receipt.',
    'E10': 'The PacketResponder thread threw an unhandled exception — indicates a serious error in the acknowledgement loop.',
    'E11': 'The PacketResponder thread for a block terminated — can be normal (end of pipeline) or abnormal (failure mid-transfer).',
    'E12': 'An exception occurred while writing a block to a mirror DataNode — the replication pipeline failed at this node.',
    'E13': 'A DataNode received an empty packet — heartbeat/end-of-stream.',
    'E14': 'Exception inside receiveBlock handler — write failed.',
    'E15': 'NameNode adjusted block offset — recovery or repair.',
    'E16': 'Block transferred successfully.',
    'E17': 'Block transfer failed.',
    'E18': 'Replication thread started.',
    'E19': 'Block reopened for append.',
    'E20': 'Delete failed due to missing metadata.',
    'E21': 'Block deleted from disk.',
    'E22': 'New block allocated by NameNode.',
    'E23': 'Block marked for deletion.',
    'E24': 'Block removed from replication queue.',
    'E25': 'Replication requested.',
    'E26': 'Block metadata updated at NameNode.',
    'E27': 'Duplicate block report received.',
    'E28': 'Orphan block reported.',
    'E29': 'Replication timed out.',
}

# ============================
# 🔹 EVENT CONTEXT
# ============================
def build_event_context_for_prompt(doc: str) -> str:
    seq_match = re.search(r'Event Sequence\s*:\s*(.+)', doc)
    if not seq_match:
        return 'No event sequence found.'

    event_ids = re.findall(r'E\d+', seq_match.group(1))
    unique_ids = list(dict.fromkeys(event_ids))

    lines = []
    for eid in unique_ids:
        desc = EVENT_DESCRIPTIONS.get(eid, 'Unknown event')
        lines.append(f'{eid}: {desc}')

    return '\n'.join(lines)

# ============================
# 🔹 SYSTEM PROMPT
# ============================
SYSTEM_PROMPT = """You are an expert in HDFS log analysis.

Return strictly valid JSON only.
"""

# ============================
# 🔹 USER PROMPT
# ============================
def compress_historical_case(doc: str, block_id: str, latency: int, distance: float) -> str:
    """Compress a full RAG doc to ~80 tokens — prevents 413 token limit errors."""
    seq_m   = re.search(r'Event Sequence\s*:\s*(.+)', doc)
    atype_m = re.search(r'Anomaly Type\(s\)\s*:\s*(.+)', doc)
    total_m = re.search(r'Total Events\s*:\s*(\d+)', doc)
    seq     = seq_m.group(1).strip()   if seq_m   else ''
    atype   = atype_m.group(1).strip() if atype_m else 'unknown'
    total   = total_m.group(1)         if total_m else '?'
    tokens  = [t.strip() for t in seq.split('->') if t.strip()]
    short   = ' -> '.join(tokens[:15])
    if len(tokens) > 15:
        short += f' (+{len(tokens)-15} more)'
    return (f'Block: {block_id} | {atype} | {latency}ms | dist={distance:.3f} | events={total}\n'
            f'Seq: {short}')


def build_user_prompt(query_doc, historical_context, query_meta):
    event_ref = build_event_context_for_prompt(query_doc)

    return f"""
QUERY BLOCK:
{query_doc}

EVENT REFERENCE:
{event_ref}

HISTORICAL CASES:
{historical_context}

Return JSON:

{{
  "block_id": "{query_meta['block_id']}",
  "summary": "...",
  "root_cause": "...",
  "comparison_to_historical": "...",
  "event_explanations": {{}},
  "anomaly_type": "...",
  "confidence": 0.0,
  "confidence_label": "medium",
  "mitigation_steps": {{
    "high": [],
    "medium": [],
    "low": []
  }}
}}

Rules:
- Never leave mitigation_steps empty.
- Provide minimum 2 steps for each confidence level that applies.
- Steps must be specific and actionable.
- Root cause must explain issue clearly + use event IDs as evidence
- event_explanations must use EVENT REFERENCE descriptions
- DO NOT return anything except JSON
"""

# ============================
# 🔥 NEW: CONFIDENCE ENGINE
# ============================
def compute_confidence(similar_docs, query_doc, latency):

    # 🔹 1. Retrieval score
    avg_distance = sum(d["distance"] for d in similar_docs) / len(similar_docs)

    if avg_distance < 0.70:
        retrieval_score = 0.9
    elif avg_distance < 0.80:
        retrieval_score = 0.7
    else:
        retrieval_score = 0.5

    # 🔹 2. Event repetition score
    event_ids = re.findall(r'E\d+', query_doc)
    counts = Counter(event_ids)

    repetition_score = sum(v for v in counts.values() if v > 1) / len(event_ids)

    # 🔹 3. Latency score
    if latency > 20000:
        latency_score = 0.9
    elif latency > 8000:
        latency_score = 0.7
    else:
        latency_score = 0.5

    # 🔥 Final confidence
    confidence = (retrieval_score + repetition_score + latency_score) / 3
    confidence = min(max(confidence, 0.0), 1.0)

    # Label
    if confidence >= 0.75:
        label = "high"
    elif confidence >= 0.4:
        label = "medium"
    else:
        label = "low"

    return round(confidence, 2), label

# ============================
# 🔹 MAIN FUNCTION (UPDATED)
# ============================
def generate_root_cause(query_doc, historical_context, query_meta, similar_docs):

    user_prompt = build_user_prompt(query_doc, historical_context, query_meta)

    try:
        response = groq_client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt},
            ],
            temperature=0.2,
            max_tokens=1500,
        )

        content = response.choices[0].message.content.strip()
        content = re.sub(r'^```(?:json)?\s*', '', content)
        content = re.sub(r'\s*```$', '', content)
        content = re.sub(r',\s*([}\]])', r'\1', content)  # strip trailing commas

        def repair_json(s: str) -> dict:
            """
            Char-by-char JSON repair covering all LLM quirks:
            1. Markdown fences   2. Trailing commas   3. Single-quoted strings
            4. Unescaped inner quotes (current bug)   5. Truncated responses
            """
            import re as _re, json as _json
            s = s.strip()
            s = _re.sub(r'^```(?:json)?\s*', '', s)
            s = _re.sub(r'\s*```$', '', s)
            s = _re.sub(r',\s*([}\]])', r'\1', s)
            # Single-quoted keys/values
            s = _re.sub(r"(?<![\w])'([^']*)'(?![\w])", r'"\1"', s)

            # Char-by-char: escape unescaped double quotes inside string values
            result    = []
            in_string = False
            i         = 0
            while i < len(s):
                c = s[i]
                if c == '\\' and in_string:
                    result.append(c)
                    i += 1
                    if i < len(s):
                        result.append(s[i])
                    i += 1
                    continue
                if c == '"':
                    if not in_string:
                        in_string = True
                        result.append(c)
                    else:
                        rest   = s[i+1:].lstrip()
                        closes = (not rest) or rest[0] in (':', ',', '}', ']')
                        if closes or i == len(s) - 1:
                            in_string = False
                            result.append(c)
                        else:
                            result.append('\\"')  # escape inner quote
                    i += 1
                    continue
                result.append(c)
                i += 1

            s = ''.join(result)
            s = _re.sub(r',\s*([}\]])', r'\1', s)

            try:
                return _json.loads(s)
            except _json.JSONDecodeError:
                # Close any open string from truncation
                if in_string:
                    s += '"'
                # Close unclosed brackets in correct inner-first order
                depth = []
                for ch in s:
                    if ch in ('{', '['):
                        depth.append('}' if ch == '{' else ']')
                    elif ch in ('}', ']') and depth:
                        depth.pop()
                s += ''.join(reversed(depth))
                s = _re.sub(r',\s*([}\]])', r'\1', s)
                return _json.loads(s)

                try:
            parsed = repair_json(content)

            # ── Enforce anomaly type ──────────────────────────────────────────
            valid_types = {'duplicate_pattern', 'repetition', 'missing_events', 'high_latency'}
            if parsed.get('anomaly_type') not in valid_types:
                parsed['anomaly_type'] = query_meta.get('primary_type', 'unknown')
                parsed['anomaly_type_corrected'] = True

        except json.JSONDecodeError as parse_err:
            # ── Print the actual error so you can see what went wrong ─────────
            print(f"[PARSE ERROR] block={query_meta.get('block_id','?')} | {parse_err}")
            print(f"  Raw content (first 300 chars): {content[:300]}")
            parsed = {
                'block_id':          query_meta.get('block_id', 'parse_error'),
                'summary':           'JSON parse failed — see printed error above.',
                'root_cause':        content[:500],
                'comparison_to_historical': '',
                'event_explanations': {},
                'anomaly_type':      query_meta.get('primary_type', 'unknown'),
                'confidence':        0.0,
                'confidence_label':  'low',
                'mitigation_steps':  {'high': [], 'medium': [], 'low': []},
                'parse_error':       True,
            }

        # ============================
        # 🔥 OVERRIDE CONFIDENCE HERE
        # ============================
        confidence, label = compute_confidence(
            similar_docs,
            query_doc,
            query_meta.get("latency", 0)
        )

        parsed["confidence"] = confidence
        parsed["confidence_label"] = label

        return parsed

    except Exception as e:
        return {
            'error': f'Groq API call failed: {str(e)}',
            'block_id': query_meta.get('block_id', 'unknown')
        }


## Step 7: Query Constructor

In [10]:
# ── Full auto-query: shows complete event sequence with no truncation ─────

def build_query_from_doc(doc: str, meta: dict) -> str:
    """
    Build the ChromaDB retrieval query from the block's parsed metadata.
    The full event sequence is included — no character-limit truncation.
    """
    anomaly_types = meta.get('anomaly_types', 'unknown')
    latency       = meta.get('latency', 0)
    total_events  = meta.get('total_events', 0)

    seq_match = re.search(r'Event Sequence\s*:\s*(.+)', doc)
    event_seq = seq_match.group(1).strip() if seq_match else ''

    query = (
        f"HDFS block anomaly: {anomaly_types}. "
        f"Latency: {latency}ms. "
        f"Total events: {total_events}. "
        f"Event sequence: {event_seq}"
    )
    return query


# Demo
sample_query = build_query_from_doc(documents[0], all_metadata[0])
print('Auto-constructed query (full, no truncation):')
print(sample_query)


Auto-constructed query (full, no truncation):
HDFS block anomaly: duplicate_pattern. Latency: 7156ms. Total events: 20. Event sequence: E5 -> E5 -> E5 -> E22 -> E11 -> E9 -> E11 -> E9 -> E11 -> E9 -> E26 -> E26 -> E26 -> E23 -> E23 -> E23 -> E21 -> E20 -> E21 -> E21


## Step 8: Single Block — End-to-End Pipeline Test

Diagnose a single block to verify the whole pipeline works correctly before batch mode.
You can change `TEST_INDEX` to try a different block.

In [11]:
TEST_INDEX = 0   # change to test a different block

query_doc  = documents[TEST_INDEX]
query_meta = all_metadata[TEST_INDEX]

print(f"Diagnosing block : {query_meta['block_id']}")
print(f"Anomaly type     : {query_meta['anomaly_types']}")
print(f"Latency          : {query_meta['latency']} ms")
print()

auto_query = build_query_from_doc(query_doc, query_meta)
print('Auto-query (full):'), print(auto_query)
print()

similar_docs = retrieve_similar_anomalies(
    query               = auto_query,
    k                   = 3,
    anomaly_type_filter = query_meta['primary_type'],
)

print('Retrieved similar cases:')
for i, r in enumerate(similar_docs):
    print(f"  Case {i+1}: block_id={r['block_id']}, "
          f"type={r['primary_type']}, latency={r['latency']}ms, distance={r['distance']:.4f}")
print()

historical_context = '\n\n'.join([
    f"Historical Case {i+1}\n"
    f"Block ID: {r['block_id']} | Latency: {r['latency']}ms | Distance: {r['distance']:.4f}\n"
    f"{r['document']}"
    for i, r in enumerate(similar_docs)
    if r['block_id'] != query_meta['block_id']
])

result = generate_root_cause(query_doc, historical_context, query_meta, similar_docs)

print('LLM Root Cause Analysis:')
print(json.dumps(result, indent=2))


Diagnosing block : blk_8462687553742484299
Anomaly type     : duplicate_pattern
Latency          : 7156 ms

Auto-query (full):
HDFS block anomaly: duplicate_pattern. Latency: 7156ms. Total events: 20. Event sequence: E5 -> E5 -> E5 -> E22 -> E11 -> E9 -> E11 -> E9 -> E11 -> E9 -> E26 -> E26 -> E26 -> E23 -> E23 -> E23 -> E21 -> E20 -> E21 -> E21

Retrieved similar cases:
  Case 1: block_id=blk_1007606047385213599, type=duplicate_pattern, latency=37507ms, distance=0.7215
  Case 2: block_id=blk_-6474543604026437248, type=duplicate_pattern, latency=37104ms, distance=0.7220
  Case 3: block_id=blk_2884813675120864357, type=duplicate_pattern, latency=37491ms, distance=0.7238

LLM Root Cause Analysis:
{
  "block_id": "blk_8462687553742484299",
  "summary": "This block contains events that repeat in a way inconsistent with the normal pipeline flow. Unlike a retry storm, the repetitions are moderate \u2014 the system was unstable but did not fully break down into a continuous retry loop.",
  "r

## Step 9: Frontend Query Examples

These are the query patterns your frontend should send to `POST /analyze`.
Three modes are supported: by `block_id`, by free-text `query`, and with an optional `anomaly_type_filter`.


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# FRONTEND QUERY EXAMPLES
# These are example payloads your frontend sends to POST /analyze
# Copy any of these into Postman / your JS fetch / your React form
# ─────────────────────────────────────────────────────────────────────────────

import json

EXAMPLE_QUERIES = [
    {
        "_description": "Mode 1: Diagnose a specific block by its ID (most common use case)",
        "payload": {
            "block_id": "blk_8462687553742484299"
        }
    },
    {
        "_description": "Mode 2: Diagnose by block ID and restrict retrieved evidence to same anomaly type",
        "payload": {
            "block_id": "blk_8462687553742484299",
            "anomaly_type_filter": "duplicate_pattern"
        }
    },
    {
        "_description": "Mode 3: Free-text query — useful when frontend user describes symptoms manually",
        "payload": {
            "query": "HDFS block with repeated E5 and E11 events, high latency around 7000ms, duplicate pipeline writes"
        }
    },
    {
        "_description": "Mode 4: Free-text query filtered to a specific anomaly type",
        "payload": {
            "query": "Block replication keeps retrying, DataNode appears stuck in a loop",
            "anomaly_type_filter": "repetition"
        }
    },
    {
        "_description": "Mode 5: Missing events — pipeline aborted before completion",
        "payload": {
            "query": "Block write pipeline aborted mid-execution, fewer than 5 events recorded, block may be corrupt",
            "anomaly_type_filter": "missing_events"
        }
    },
    {
        "_description": "Mode 6: High latency — block completed but took too long",
        "payload": {
            "query": "Block pipeline completed successfully but latency was over 30000ms, possible disk or network bottleneck",
            "anomaly_type_filter": "high_latency"
        }
    },
    {
        "_description": "Mode 7: Retrieve more historical cases (k=5) for a deeper analysis",
        "payload": {
            "block_id": "blk_8462687553742484299",
            "k": 5
        }
    },
]

print("=" * 65)
print("FRONTEND QUERY EXAMPLES — send these as POST /analyze body")
print("=" * 65)
for ex in EXAMPLE_QUERIES:
    print(f"\n{ex['_description']}")
    print(json.dumps(ex['payload'], indent=2))

print()
print("=" * 65)
print("JS fetch example (paste into your frontend):")
print("=" * 65)
print("""
const response = await fetch('YOUR_CLOUDFLARE_URL/analyze', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({
    block_id: 'blk_8462687553742484299'   // or use query: 'your description'
  })
});
const result = await response.json();
console.log(result);
""")


FRONTEND QUERY EXAMPLES — send these as POST /analyze body

Mode 1: Diagnose a specific block by its ID (most common use case)
{
  "block_id": "blk_8462687553742484299"
}

Mode 2: Diagnose by block ID and restrict retrieved evidence to same anomaly type
{
  "block_id": "blk_8462687553742484299",
  "anomaly_type_filter": "duplicate_pattern"
}

Mode 3: Free-text query — useful when frontend user describes symptoms manually
{
  "query": "HDFS block with repeated E5 and E11 events, high latency around 7000ms, duplicate pipeline writes"
}

Mode 4: Free-text query filtered to a specific anomaly type
{
  "query": "Block replication keeps retrying, DataNode appears stuck in a loop",
  "anomaly_type_filter": "repetition"
}

Mode 5: Missing events — pipeline aborted before completion
{
  "query": "Block write pipeline aborted mid-execution, fewer than 5 events recorded, block may be corrupt",
  "anomaly_type_filter": "missing_events"
}

Mode 6: High latency — block completed but took too long
{


## Step 10: Batch Analysis

Run the full pipeline over a sample of blocks and save results to CSV.
Adjust `SAMPLE_SIZE` and `SAMPLE_PER_TYPE` as needed.

In [13]:
import time

# ── Balanced sample: pick N blocks per anomaly type ───────────────────────────
SAMPLE_PER_TYPE = 5   # blocks to diagnose per anomaly type
SLEEP_BETWEEN   = 3.0 # seconds between Groq calls (rate-limit safety)

meta_df = pd.DataFrame(all_metadata)
meta_df["doc_index"] = meta_df.index

sampled_indices = (
    meta_df
    .groupby("primary_type", group_keys=False)
    .apply(lambda g: g.sample(min(SAMPLE_PER_TYPE, len(g)), random_state=42), include_groups=False)
    ["doc_index"]
    .tolist()
)

print(f"Blocks to diagnose: {len(sampled_indices)}")

batch_results = []

for idx in sampled_indices:
    q_doc  = documents[idx]
    q_meta = all_metadata[idx]

    print(f"[{idx}] {q_meta['block_id']} | {q_meta['primary_type']}")

    auto_query = build_query_from_doc(q_doc, q_meta)

    similar = retrieve_similar_anomalies(
        query               = auto_query,
        k                   = 3,
        anomaly_type_filter = q_meta["primary_type"],
    )

    hist_ctx = "\n\n".join([
        f"Case {i+1}\n" +
        compress_historical_case(r['document'], r['block_id'], r['latency'], r['distance'])
        for i, r in enumerate(similar)
        if r["block_id"] != q_meta["block_id"]
    ])

    #llm_result = generate_root_cause(q_doc, hist_ctx)
    llm_result = generate_root_cause(q_doc, hist_ctx, q_meta, similar)

    # Flatten for CSV export
    batch_results.append({
        "block_id":               q_meta["block_id"],
        "primary_type":           q_meta["primary_type"],
        "latency_ms":             q_meta["latency"],
        "llm_block_id":           llm_result.get("block_id", ""),
        "summary":                llm_result.get("summary", ""),
        "root_cause":             llm_result.get("root_cause", ""),
        "comparison_to_historical": llm_result.get("comparison_to_historical", ""),
        "anomaly_type_llm":       llm_result.get("anomaly_type", ""),
        "confidence":             llm_result.get("confidence", 0.0),
        "confidence_label":       llm_result.get("confidence_label", ""),
        "mitigation_high":        " | ".join(s for s in llm_result.get("mitigation_steps", {}).get("high", [])   or [] if s),
        "mitigation_medium":      " | ".join(s for s in llm_result.get("mitigation_steps", {}).get("medium", []) or [] if s),
        "mitigation_low":         " | ".join(s for s in llm_result.get("mitigation_steps", {}).get("low", [])    or [] if s),
        "error":                  llm_result.get("error", ""),
    })

    time.sleep(SLEEP_BETWEEN)

batch_df = pd.DataFrame(batch_results)

out_path = "/content/drive/MyDrive/Log_Anamoly_Detection/results/llm_root_cause_results.csv"
batch_df.to_csv(out_path, index=False)

print(f"\nSaved {len(batch_df)} results to {out_path}")
print(batch_df[["block_id", "primary_type", "confidence_label", "summary"]].to_string())

Blocks to diagnose: 20
[1314] blk_-2380425631987325831 | duplicate_pattern


/tmp/ipykernel_5513/3074907348.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(SAMPLE_PER_TYPE, len(g)), random_state=42))


[2430] blk_8687300722271711987 | duplicate_pattern
[3275] blk_8012251783063320280 | duplicate_pattern
[2604] blk_1144859527738276881 | duplicate_pattern
[2239] blk_8686401382491198120 | duplicate_pattern
[3153] blk_8321243236616059836 | high_latency
[1744] blk_8520238754901860870 | high_latency
[2033] blk_8722120357524151345 | high_latency
[3058] blk_6240020657057666020 | high_latency
[2613] blk_-2601198341395425767 | high_latency
[1621] blk_-3797243824001506600 | missing_events
[1969] blk_-6554481336966240622 | missing_events
[2549] blk_-2960077722173004125 | missing_events
[3168] blk_6604728773479221839 | missing_events
[1141] blk_-4969303299923717599 | missing_events
[2957] blk_8717072267468231918 | repetition
[1619] blk_8398325026949583934 | repetition
[2912] blk_-3800595472166056394 | repetition
[1898] blk_-1021695699039097568 | repetition
[PARSE ERROR] block=blk_-1021695699039097568 | Expecting ',' delimiter: line 5 column 103 (char 736)
  Raw content (first 300 chars): {
  "bloc